In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import (
    PROJECT_ROOT,
    DATA_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    WEATHER_DIR,
    SOIL_DIR,
    RAW_DIR,
    OUTPUT_DIR,
    PROCESSED_DATASET,
    WEATHER_FEATHER,
    MERGED_DATA_DIR,
    interim_csb_path,
)


# Rotation Strategy Classification

This notebook focuses **exclusively on rotation strategy prediction** — classifying each NY agricultural polygon into one of three multi-year strategy categories:

| Label | Meaning |
|---|---|
| **Continuous** | Same crop grown in all observed years (e.g., Continuous Corn) |
| **Rotation** | Exactly two crops alternated over the observation window (e.g., Corn-Soy Rotation) |
| **Complex/Mixed** | Three or more distinct crops in the sequence |

---


## Why rotation strategy vs. next-crop prediction?

The companion notebook `analysis_notebook.ipynb` predicts the **next year's crop** for each polygon — a point-in-time classification task. This notebook addresses a **coarser, multi-year question**: what is the underlying rotation *strategy* a farmer follows?

- **More policy-relevant**: rotation strategy links directly to soil health, nitrogen management, and conservation program eligibility.
- **Less noisy target**: year-to-year crop choice contains weather-driven deviations; the strategy label averages over these.
- **Longer temporal horizon**: features extend to Lag-5 (5-year crop history) to capture full rotation cycles.

---

## Key differences from `analysis_notebook.ipynb`

| Aspect | `analysis_notebook.ipynb` | This notebook |
|---|---|---|
| **Target variable** | Next-year crop (5 classes) | Multi-year strategy (3 classes) |
| **Granularity** | One row per polygon × year | One row per polygon |
| **Lag features** | Lag-1, Lag-2 | Lag-1 through Lag-5 |
| **Weather** | Year-specific values | County-level mean ± std across 2008-2024 |
| **Train/test split** | By year (≤2022 train) | Random 80/20 on polygons |
| **Neural network** | Geometric CNN + RNN | LSTM sequence classifier |

---

## Notebook Structure

1. **Data Exploration** — Load CSB data, assign strategy labels, visualize class balance and spatial distribution
2. **Feature Engineering** — Build polygon-level features: extended lags, climate normals, rotation pattern indicators
3. **Baseline Models** — KNN and CatBoost on the 3-class target
4. **Neural Network** — LSTM sequence classifier reading the full 17-year crop sequence

## 1. Data Exploration

**Script:** `src/rot_strategy_processing.py`

**What this step does:**
- Loads all three CSB feather files (2008-2015, 2016-bridge, 2017-2024) using the same three-source strategy as the companion notebook.
- Merges them into a single wide table: one row per polygon with columns `CDL2008 … CDL2024`.
- Classifies each polygon's multi-year crop sequence into a `Rotation_Type` (e.g., `Rot Corn-Soybeans`) and a 3-class `Strategy_Category`.
- Saves `output/rot_strategy_labeled.parquet` for downstream scripts.

**Outputs:**
- `output/rot_strategy_top15.png` — Top-15 rotation strategies by total acreage
- `output/rot_strategy_categories.png` — 3-class category totals
- `output/rot_strategy_class_balance.png` — Pie chart of polygon counts
- `output/rot_strategy_spatial_map.png` — Spatial distribution across NY

In [ ]:
%run src/rot_strategy_processing.py

## 2. Feature Engineering

**Script:** `src/rot_strategy_feature_engineering.py`

**What this step does:**

Builds a **polygon-level** (one row per polygon) feature table from:

### Rotation History Features (Lag-5)
- `Crop_Lag1` … `Crop_Lag5` — CDL codes of the 5 most recent crops grown
- `Crop_Type_Lag1` … `Crop_Type_Lag5` — 5-class agronomic categories
- `Crop_Diversity_L5` — number of unique crops in the last 5 years (higher = more complex)
- `Crop_Changed_L1_L2` — binary: did the crop change between the last 2 years?
- `Is_Alternating_L1_L3` — binary: is the polygon in a strict alternating pattern?
- `Continuity_Streak` — how many consecutive years the most recent crop was repeated

### Climate Normals (Time-Averaged Weather)
- `Mean_Planting_Precip` — county average April-May precipitation across 2008-2024
- `Std_Planting_Precip` — inter-annual variability in planting precipitation
- `Mean_Growing_GDD` — county average May-Oct growing degree days
- `Std_Growing_GDD` — inter-annual variability in growing degree days

### County Context
- `County_Dominant_Strategy` — most common strategy category in the county
- `County_Crop_Diversity` — mean number of distinct crops per polygon in the county
- `County_Avg_Field_Size` — mean CSBACRES per county

### Spatial
- `Longitude_Norm`, `Latitude_Norm` — standardised polygon centroid coordinates
- `CNTYFIPS` — county identifier

**Output:** `output/rot_strategy_features.parquet`

In [ ]:
%run src/rot_strategy_feature_engineering.py

## 3. Baseline Models

**Script:** `src/rot_strategy_model_baseline.py`

**Target:** `Strategy_Category` — 3 classes (Continuous / Rotation / Complex/Mixed)

**Train/test split:** Random stratified 80/20 on polygons (not by year, since the label is polygon-level).

### Models

1. **KNN (k=5)** — StandardScaler preprocessing; stratified subsample of 30,000 for efficiency. Captures regional clustering: polygons near each other tend to follow similar strategies.

2. **CatBoost** — 100 iterations, depth=4, `auto_class_weights='Balanced'` to address the class imbalance. Passes `County_Dominant_Strategy` as a native categorical feature.

### Outputs
- Confusion matrices (3×3) per model
- ROC curves (one-vs-rest for each strategy class)
- Feature importance bar chart (CatBoost)
- `output/rot_strategy_model_comparison.csv`

In [ ]:
%run src/rot_strategy_model_baseline.py

## 4. Neural Network — LSTM Sequence Classifier

**Script:** `src/rot_strategy_model_advanced.py`

Rotation strategy is inherently a **sequence classification** problem: the LSTM reads the ordered crop sequence `[CDL2008, CDL2009, …, CDL2024]` for each polygon and predicts `Strategy_Category`.

### Architecture

```
Input:  (batch, T=17, features=3)
        features = [crop_code_normalised, planting_precip, growing_gdd]
   ↓
LSTM:   hidden_size=64, num_layers=2, dropout=0.3
   ↓  (final hidden state)
Dropout(0.3)
   ↓
Linear: 64 → 3
   ↓
Softmax → Strategy_Category
```

### Why LSTM for strategies?
- Captures **long-range temporal dependencies**: whether a polygon switched strategies mid-period is visible only in the full 17-year sequence.
- Handles **variable-length histories**: polygons with fewer observed years are zero-padded.
- The **final hidden state** encodes the cumulative pattern, unlike lag features that only see the last 5 years.

### Training Configuration
| Hyperparameter | Value |
|---|---|
| Hidden size | 64 |
| Layers | 2 |
| Dropout | 0.3 |
| Learning rate | 0.001 |
| Epochs | 50 |
| Batch size | 64 |
| Sample size | 50,000 polygons |
| Loss | CrossEntropyLoss with class weights |

### Outputs
- `output/rot_strategy_lstm_loss_curve.png` — training/validation loss
- `output/rot_strategy_lstm_confusion.png` — confusion matrix
- `output/rot_strategy_lstm_results.csv` — accuracy and F1, appended to comparison table

In [ ]:
%run src/deep_rot_strategy.py

## Model Evaluation Summary

After running all four sections, compare the models using the saved CSVs:

- `output/rot_strategy_model_comparison.csv` — KNN and CatBoost metrics
- `output/rot_strategy_lstm_results.csv` — LSTM metrics

The cell below loads and displays both together.

In [ ]:
import os
import pandas as pd

baseline_path = os.path.join("output", "rot_strategy_model_comparison.csv")
lstm_path     = os.path.join("output", "rot_strategy_lstm_results.csv")

frames = []
if os.path.exists(baseline_path):
    frames.append(pd.read_csv(baseline_path))
if os.path.exists(lstm_path):
    frames.append(pd.read_csv(lstm_path))

if frames:
    summary = pd.concat(frames, ignore_index=True)
    display(summary)
else:
    print("No results found yet — run the sections above first.")

In [ ]:
# 5. Export per-polygon strategy predictions for acreage regression
# -----------------------------------------------------------------
# This cell creates a polygon-level prediction table with class probabilities.
# Leakage guard: County_Dominant_Strategy is excluded from model features.

import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

INPUT_FEATURES = os.path.join("output", "rot_strategy_features.parquet")
OUTPUT_PRED    = os.path.join("output", "rot_strategy_polygon_predictions.parquet")

STRATEGY_MAPPING = {"Continuous": 0, "Rotation": 1, "Complex/Mixed": 2}
STRATEGY_NAMES   = {v: k for k, v in STRATEGY_MAPPING.items()}

if not os.path.exists(INPUT_FEATURES):
    raise FileNotFoundError(f"{INPUT_FEATURES} not found. Run feature engineering cells first.")

pred_df = pd.read_parquet(INPUT_FEATURES).copy()
pred_df = pred_df.dropna(subset=["CSBID", "Strategy_Category"]).copy()
pred_df["Target"] = pred_df["Strategy_Category"].map(STRATEGY_MAPPING)
pred_df = pred_df.dropna(subset=["Target"]).copy()
pred_df["Target"] = pred_df["Target"].astype(int)

for i in range(1, 6):
    c = f"Crop_Lag{i}"
    if c in pred_df.columns:
        pred_df[c] = pd.to_numeric(pred_df[c], errors="coerce")

numeric_candidates = [
    "Crop_Lag1", "Crop_Lag2", "Crop_Lag3", "Crop_Lag4", "Crop_Lag5",
    "Crop_Diversity_L5", "Crop_Changed_L1_L2", "Is_Alternating_L1_L3", "Continuity_Streak",
    "Mean_Planting_Precip", "Std_Planting_Precip", "Mean_Growing_GDD", "Std_Growing_GDD",
    "Longitude_Norm", "Latitude_Norm", "County_Crop_Diversity", "County_Avg_Field_Size", "CNTYFIPS",
]

# Leakage guard: do not use County_Dominant_Strategy.
feature_cols = [c for c in numeric_candidates if c in pred_df.columns]
for c in feature_cols:
    pred_df[c] = pd.to_numeric(pred_df[c], errors="coerce")
    pred_df[c] = pred_df[c].fillna(pred_df[c].median())

X_all = pred_df[feature_cols]
y_all = pred_df["Target"]

if CATBOOST_AVAILABLE:
    clf = CatBoostClassifier(
        iterations=150,
        learning_rate=0.08,
        depth=5,
        random_state=42,
        verbose=False,
        auto_class_weights="Balanced",
    )
    clf.fit(X_all, y_all)
    y_pred = clf.predict(X_all).astype(int).ravel()
    y_proba = clf.predict_proba(X_all)
    model_name = "CatBoost"
else:
    pre = ColumnTransformer([("num", StandardScaler(), feature_cols)])
    clf = Pipeline([
        ("preprocessor", pre),
        ("classifier", KNeighborsClassifier(n_neighbors=7, n_jobs=-1)),
    ])
    clf.fit(X_all, y_all)
    y_pred = clf.predict(X_all).astype(int)
    y_proba = clf.predict_proba(X_all)
    model_name = "KNN"

pred_out = pred_df[["CSBID", "CNTYFIPS", "Strategy_Category", "Rotation_Type"]].copy()
pred_out["Predicted_Strategy_Code"] = y_pred
pred_out["Predicted_Strategy_Label"] = pd.Series(y_pred).map(STRATEGY_NAMES)
pred_out["p_Continuous"] = y_proba[:, 0]
pred_out["p_Rotation"] = y_proba[:, 1]
pred_out["p_ComplexMixed"] = y_proba[:, 2]
pred_out["Prediction_Model"] = model_name

pred_out.to_parquet(OUTPUT_PRED, index=False)

print(f"Prediction model: {model_name}")
print(f"In-sample accuracy (quick diagnostic): {accuracy_score(y_all, y_pred):.4f}")
print(f"Saved per-polygon predictions: {OUTPUT_PRED}")
print(f"Rows: {len(pred_out):,} | Columns: {list(pred_out.columns)}")

In [ ]:
# verify if the parquet data are exported correctly
exp_data = pd.read_parquet("output/rot_strategy_polygon_predictions.parquet")
exp_data.head()

# exp_data1 = pd.read_parquet("output/rot_strategy_labeled.parquet")
# exp_data1.head()

# exp_data2 = pd.read_parquet("output/rot_strategy_features.parquet")
# exp_data2.head()


In [ ]:
# 6. Export merged datasets for acreage regression
# -----------------------------------------------
# Outputs:
#   1) Wide polygon table: classification + geometric data
#   2) Polygon-year table: year-specific weather + soil (no lag columns) + strategy outputs

import os
import pandas as pd

PRED_PATH = os.path.join("output", "rot_strategy_polygon_predictions.parquet")
LABELED_PATH = os.path.join("output", "rot_strategy_labeled.parquet")
FEAT_PATH = os.path.join("output", "rot_strategy_features.parquet")

OUT_WIDE = os.path.join("output", "regression_polygon_wide_strategy_geometric.parquet")
OUT_LONG = os.path.join("output", "regression_polygon_year_nolag_strategy.parquet")

if not os.path.exists(PRED_PATH):
    raise FileNotFoundError(f"{PRED_PATH} not found. Run the previous export cell first.")

pred = pd.read_parquet(PRED_PATH)
labeled = pd.read_parquet(LABELED_PATH) if os.path.exists(LABELED_PATH) else None
feat = pd.read_parquet(FEAT_PATH) if os.path.exists(FEAT_PATH) else None

# -----------------------
# (A) Wide polygon export
# -----------------------
base_cols = ["CSBID", "CNTYFIPS", "CSBACRES", "INSIDE_X", "INSIDE_Y"]
geo_cols = ["CSBID", "Longitude_Norm", "Latitude_Norm"]

if labeled is not None:
    wide_base = labeled[[c for c in base_cols if c in labeled.columns]].drop_duplicates("CSBID")
else:
    wide_base = pred[["CSBID", "CNTYFIPS"]].drop_duplicates("CSBID")

if feat is not None:
    geo = feat[[c for c in geo_cols if c in feat.columns]].drop_duplicates("CSBID")
    wide_base = wide_base.merge(geo, on="CSBID", how="left")

wide_out = wide_base.merge(pred, on=[c for c in ["CSBID", "CNTYFIPS"] if c in wide_base.columns and c in pred.columns], how="left")

# Explicit leakage guard requested by plan/user.
if "County_Dominant_Strategy" in wide_out.columns:
    wide_out = wide_out.drop(columns=["County_Dominant_Strategy"])

wide_out.to_parquet(OUT_WIDE, index=False)
print(f"Saved wide polygon dataset: {OUT_WIDE}")
print(f"Shape: {wide_out.shape}")

# ------------------------------------------------------
# (B) Polygon-year export with no weather lag features
# ------------------------------------------------------
# Prefer processed dataset generated by src/feature_engineering.py
candidate_processed_paths = [
    str(PROCESSED_DATASET),
    os.path.join("data", "processed_dataset.parquet"),
    "processed_dataset.parquet",
]
processed_path = next((p for p in candidate_processed_paths if os.path.exists(p)), None)

if processed_path is None:
    print("processed_dataset.parquet not found; skipped polygon-year export.")
else:
    panel = pd.read_parquet(processed_path)

    # Remove all lagged columns (weather/crop) per request.
    lag_cols = [c for c in panel.columns if "Lag" in c or "_Lag" in c]
    panel = panel.drop(columns=lag_cols, errors="ignore")

    # Ensure strategy is merged by polygon ID and leakage column is dropped.
    strategy_cols = [
        "CSBID", "Predicted_Strategy_Code", "Predicted_Strategy_Label",
        "p_Continuous", "p_Rotation", "p_ComplexMixed", "Strategy_Category", "Rotation_Type",
    ]
    strategy_cols = [c for c in strategy_cols if c in pred.columns]

    merged = panel.merge(pred[strategy_cols], on="CSBID", how="left")
    merged = merged.drop(columns=["County_Dominant_Strategy"], errors="ignore")

    merged.to_parquet(OUT_LONG, index=False)
    print(f"Saved polygon-year no-lag dataset: {OUT_LONG}")
    print(f"Source panel: {processed_path}")
    print(f"Shape: {merged.shape}")